# <font color="#49699E" size=40>Processing Natural Language Data</font>

# LEARNING OBJECTIVES
# LEARNING MATERIALS
# INTRODUCTION
## Package Imports

In [1]:
import pandas as pd
pd.set_option("display.notebook_repr_html", False)
import seaborn as sns
import matplotlib.pyplot as plt

from dcss.plotting import format_axes_commas, custom_seaborn
from dcss.text import bigram_process, preprocess

import spacy
from spacy import displacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

custom_seaborn()

# TEXT PROCESSING


## Getting to Know SpaCy


In [2]:
nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

### The SpaCy NLP Pipeline


### The SpaCy Containers


In [3]:
with open('../data/txt_files/bonikowski_2017.txt', 'r') as f:
    abstract = f.read()

#### `Doc`s


In [4]:
doc = nlp(abstract)
print(f'There are {len(doc)} tokens in this document.')

There are 346 tokens in this document.


In [5]:
from spacy.tokens import DocBin

doc_export = DocBin()
doc_export.add(doc)
doc_export.to_disk('../data/misc/bart_bonikowski_doc.spacy')

In [6]:
doc_import = DocBin().from_disk('../data/misc/bart_bonikowski_doc.spacy')
docs = list(doc_import.get_docs(nlp.vocab))
doc = docs[0]
print(f'There are {len(doc)} tokens in this document.')

There are 346 tokens in this document.


#### `Token`
##### `Span`


# NORMALIZING TEXT VIA LEMMATIZATION


In [7]:
nlp = spacy.load('en_core_web_sm', disable=['ner'], exclude = ['lemmatizer'])
lemmatizer = nlp.add_pipe('lemmatizer', config = {'mode': 'rule'})
lemmatizer.initialize()

In [8]:
doc = nlp(abstract)
lemmatized = [(token.text, token.lemma_) for token in doc]

In [9]:
for each in lemmatized[:100]:
    if each[0].lower() != each[1].lower():
        print(f'{each[0]} ({each[1]})')

accounts (account)
successes (success)
politics (politic)
including (include)
phenomena (phenomenon)
are (be)
elements (element)
are (be)
limited (limit)
resulting (result)
has (have)
hindered (hinder)
accounts (account)
causes (cause)
consequences (consequence)
existing (exist)


# PART-OF-SPEECH TAGGING


In [10]:
for item in doc[:20]:
    print(f'{item.text} ({item.pos_})')

Scholarly (ADJ)
and (CCONJ)
journalistic (ADJ)
accounts (NOUN)
of (ADP)
the (DET)
recent (ADJ)
successes (NOUN)
of (ADP)
radical (ADJ)
- (PUNCT)
right (NOUN)
politics (NOUN)
in (ADP)
Europe (PROPN)
and (CCONJ)
the (DET)
United (PROPN)
States (PROPN)
, (PUNCT)


In [11]:
nouns = [item.text for item in doc if item.pos_ == 'NOUN']
print(nouns[:20])

['accounts', 'successes', 'right', 'politics', 'referendum', 'campaign', 'phenomena', 'populism', 'ethno', 'nationalism', 'authoritarianism', 'elements', 'right', 'right', 'lack', 'clarity', 'accounts', 'causes', 'consequences', 'ethno']


In [12]:
adjectives = [item.text for item in doc if item.pos_ == 'ADJ']
adjectives[:20]

['Scholarly',
 'journalistic',
 'recent',
 'radical',
 'important',
 'radical',
 'coterminous',
 'analytical',
 'nationalist',
 'contemporary',
 'temporal',
 'discursive',
 'corresponding',
 'public',
 'available',
 'radical',
 'stable',
 'public',
 'radical',
 'pre']

In [13]:
parts = ['NOUN', 'ADJ']
words = [item.text for item in doc if item.pos_ in parts]
words[:20]

['Scholarly',
 'journalistic',
 'accounts',
 'recent',
 'successes',
 'radical',
 'right',
 'politics',
 'referendum',
 'campaign',
 'phenomena',
 'populism',
 'ethno',
 'nationalism',
 'authoritarianism',
 'important',
 'elements',
 'radical',
 'right',
 'coterminous']

# SYNTACTIC DEPENDENCY PARSING


In [14]:
sentence = nlp("This book is a practical guide to computational social science")

## Noun Chunks 


In [15]:
for item in list(doc.noun_chunks)[:10]:
    print(item.text)

Scholarly and journalistic accounts
the recent successes
radical-right politics
Europe
the United States
the Brexit referendum
the Trump campaign
three phenomena
populism
ethno-nationalism


## Extracting Words by Dependency Labels:  Subject, Verb, Object Triplets


In [16]:
for sent in doc.sents:
    tvdo = [(token.head.text, token.text) for token in sent if token.dep_ == 'dobj']
    print(tvdo)

[('conflate', 'phenomena')]
[]
[('hindered', 'accounts')]
[('address', 'problem'), ('bring', 'research'), ('define', 'concepts'), ('examine', 'patterns'), ('bring', 'strategies')]
[('understand', 'support')]
[('shifting', 'context')]
[('engendered', 'sense')]
[('channelled', 'threats'), ('activating', 'attitudes'), ('lending', 'legitimacy'), ('return', 'power')]
[('threaten', 'institutions'), ('has', 'potential'), ('alter', 'contours'), ('creating', 'conditions')]


In [17]:
from dcss.svo import subject_verb_object_triples

list(subject_verb_object_triples(doc))

[(accounts, tend, to conflate),
 (lack, has hindered, accounts),
 (I, bring, research),
 (I, bring, strategies),
 (I, bring, attitudes),
 (variety, have engendered, sense),
 (discourse, has channelled, threats),
 (that, promise, to return),
 (form, threaten, institutions),
 (form, threaten, -group relations),
 (it, has, potential)]

# CONCLUSION
## Key Points 
